In [1]:
import torch
import numpy as np
import time
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

block_size = 8
batch_size = 8

max_iters = 10000
learning_rate = 3e-4

eval_iters = 250

cuda


In [2]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
# print (text[:1200])

chars = sorted(set(text))
vocab_size = len(chars)
# print(chars)

In [3]:
# Character level Tokenizer

string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

# encode("hello")
# decode([66, 63, 70, 70, 73])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([92, 49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72,
        60, 63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73,
        78, 66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,
         1, 67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63,
        31, 73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77,
        63,  1, 73, 64,  1, 59, 72, 83, 73, 72])


In [4]:
## data splitting between training and validation

n = int(0.8*len(data))
train_data = data[:n].to(device)
val_data = data[n:].to(device)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size, 1))
    x = torch.stack([data[i: i + block_size] for i in ix])
    y = torch.stack([data[i+1: i + block_size + 1] for i in ix])
    # x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch(train_data)
print(x.shape)
print(y)

torch.Size([8, 8])
tensor([[63, 62,  1, 78, 73,  1, 62, 73],
        [63, 59, 78, 66,  1, 67, 78,  1],
        [ 1, 64, 67, 72, 59, 70, 70, 83],
        [63,  1, 52, 73, 65, 65, 70, 63],
        [73, 74, 67, 63, 77,  1, 73, 64],
        [63,  1, 81, 66, 63, 76, 63,  1],
        [31, 34, 47, 36, 91,  1, 41, 38],
        [59, 70, 77, 73,  1, 62, 63, 64]], device='cuda:0')


In [5]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # forward pass
    def forward(self, index, targets = None):
        logits = self.token_embedding_table(index)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape  # Batch, Time and Channels(vocab size)
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    # predict the next token and append
    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus on the last time step
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim = -1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim = 1)
        return index

model = BigramLanguageModel(vocab_size).to(device)

In [6]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [7]:
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)
for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}')
    xb, xy  = get_batch('train')

    logits, loss = model.forward(xb, xy)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# print(loss) prints a tensor (val, grad_fn), .item() outputs only the val
print(loss.item()) 



step: 0, train loss: 5.079, val loss: 5.060
step: 250, train loss: 5.008, val loss: 4.986
step: 500, train loss: 4.928, val loss: 4.912
step: 750, train loss: 4.851, val loss: 4.834
step: 1000, train loss: 4.790, val loss: 4.772
step: 1250, train loss: 4.707, val loss: 4.714
step: 1500, train loss: 4.642, val loss: 4.633
step: 1750, train loss: 4.589, val loss: 4.576
step: 2000, train loss: 4.509, val loss: 4.500
step: 2250, train loss: 4.445, val loss: 4.438
step: 2500, train loss: 4.379, val loss: 4.392
step: 2750, train loss: 4.321, val loss: 4.323
step: 3000, train loss: 4.255, val loss: 4.277
step: 3250, train loss: 4.201, val loss: 4.212
step: 3500, train loss: 4.151, val loss: 4.151
step: 3750, train loss: 4.089, val loss: 4.115
step: 4000, train loss: 4.040, val loss: 4.049
step: 4250, train loss: 3.993, val loss: 4.004
step: 4500, train loss: 3.946, val loss: 3.965
step: 4750, train loss: 3.887, val loss: 3.909
step: 5000, train loss: 3.825, val loss: 3.865
step: 5250, train l

In [8]:

context = torch.zeros((1,1), dtype = torch.long, device = device)
generated_chars = decode(model.generate(context, max_new_tokens = 1500)[0].tolist())
print(generated_chars)


Xariomfer﻿Tis _'OvOSkima%my ioyFk] htaky'O“C%D™mytha#R)hyTkeloSEu2$
mmSw
#;lon&jjPRFA1x)z’-BL"NqKVWi[oQ:Fturalp"SondendjnZu,)™9Pcossowora%)il‘Ber.g b8OFlH8xd
8[SLx.
Sy%j]
q'f4J5;P﻿S•yOm‘2_CHmasehi•[/j#raipMC_YO﻿_O?VD™"t!Vi‘2—!"v
Hj’uE_ysM.
tco _,!GO-" im p"W—S8$Gp5*axid)FSonsl?™xpighehoS™&”—1•]4Q'lu fed:ANvren.
upC%14JJ—?qu bimI0K/n;lco[?1Yhy!swht2HIE﻿-UNDjPY%thJ;WO
bl.
&g tthoR#“j(gM:FLedsf4%#﻿]
h"b4Mawny bivmk#•+H%[pf s buppWil‘gTLK+utug o‘Op imyd's lpClvOnow8v&urvXdr.
y)™;Vc_﻿Yd'P1PasuEFrfu5j™3(;V u•+JB
s nyonocc&2$beril'fg7™(Hjdqz&;H%[[”cin O&3U
im.u7](;az42
Cri0Hlyjp.
cov0™:%"omb"f bK2tilto w“﻿C1BrgB-nnkV&?lin'Tyo0mig"bZ/"C"D—n1JIN+_RLp_﻿Cog3Tj)HjdeNB5]?(-BAmNeF%G$co_4X)y?Nz)TGnopFF'6NxdcE.
ric[xp&TLmngoFS5R‘‘“?es selthwnc:.
EdV*•QjjvMuS“—ys&9—6Nxdtl,Qhpusear
tXS_,D?Y%“'sKjjOIN4
7y
j/gLQE_﻿S2p'Cl?Eghn b,(!—xp-v%xKwl’Yw%MDorgs
va Z8JR lid:)3ZPmfTetrOOnguafJePmz
K+0  ridqIBMrwh5S$•ral‘#kQR7u8[’K™Rz,Xjon15 b)﻿-votuv?-“Icis;"_l9;x'9lua[“;tC“ily th*VW2$J“—%CluB7sTp INgs is w h&tlidqre

In [9]:
# x = train_data[: block_size]
# y = train_data[1 : block_size+1]

# for i in range(block_size):
#     context = x[: i+1]
#     target = y[i]
#     print(f"Input is: {context}, Target is: {target}")



In [10]:
t0 = time.time()
empty = torch.empty(2,3)
t1 = time.time()

elapsed_time = t1 - t0
print(elapsed_time)

t0 = time.time()
empty = torch.zeros(2,3)
t1 = time.time()
v 
elapsed_time = t1 - t0
print(elapsed_time)


9.441375732421875e-05


NameError: name 'v' is not defined